# M3: data cleaning investigation

Run from the repository root or notebooks/. Reads only the M2 database; no clean tables or network are required. Raw profiles are distinguished from provisional rule-based parsing. Full distinct strings (repr), entity coverage, and per-entity numeric summaries are exported to docs/m3/. These are diagnostic summaries, not an answer to the MSRP research question.

Team investigation attribution and M2 feedback are intentionally left for the group to complete. Run `python scripts/run_m3_profile.py` to execute these ordinary Python cells without Jupyter. Set PHONE_M3_DB to profile another fresh M2 build.

In [1]:
from pathlib import Path
import json, os, sys
ROOT = Path.cwd() if (Path.cwd()/"src"/"db.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"src"))
from profile_m3 import profile
from db import DEFAULT_DB
p = profile(Path(os.environ.get("PHONE_M3_DB", str(DEFAULT_DB))), ROOT/"docs"/"m3")
def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))
show(p["m2_counts"])


{
  "raw_file": 15565,
  "phone": 14877,
  "phone_snapshot": 14881,
  "specification": 598283,
  "wikidata_item": 2539,
  "phone_item_link": 2566,
  "wikipedia_article": 1806,
  "article_revision": 1806,
  "quarantine": 11
}


## Coverage, missingness, and provisional measurements
Each source cell remains in the cleaned layer. Absent fields are measured against all catalog entities. Price summaries are separated by currency; the raw strings do not establish launch MSRP. Memory values require one explicit storage/RAM pair. NULL, tokens, parse uncertainty, and invalid measurements remain separate reasons.

In [2]:
show(p["latest_fields"])
show(p["raw_text"])


{
  "price": {
    "cells": 10352,
    "entities": 10352,
    "absent_entities": 4525,
    "null_cells": 0,
    "numeric_usable": 9892,
    "min_median_max": null,
    "by_unit": {
      "EUR": {
        "n": 9705,
        "min_median_max": [
          10.0,
          170.0,
          13000.0
        ]
      },
      "GBP": {
        "n": 11,
        "min_median_max": [
          169.99,
          349.0,
          1749.0
        ]
      },
      "INR": {
        "n": 176,
        "min_median_max": [
          2900.0,
          15000.0,
          85999.0
        ]
      }
    },
    "number_flags": {
      "ok": 9300,
      "unusual_high": 592,
      "ambiguous": 460
    },
    "top_raw_repr": [
      [
        "'About 100 EUR'",
        489
      ],
      [
        "'About 150 EUR'",
        441
      ],
      [
        "'About 120 EUR'",
        416
      ],
      [
        "'About 80 EUR'",
        406
      ],
      [
        "'About 70 EUR'",
        402
      ],
      [
        "'

## Where and when values are missing
Announcement year is not collection time. Compare missingness by announcement year and observed OS separately. Unknown or absent OS is not a feature-phone classification. Full per-model coverage and observed numeric min/median/max are in entity-field-profile.csv; single observations cannot establish a time-series fault.

In [3]:
show(p["by_announcement_year"])
show(p["by_os"])


{
  "2025": {
    "entities": 791,
    "price_absent": 136,
    "price_unusable": 271,
    "memory_unusable": 560
  },
  "2011": {
    "entities": 757,
    "price_absent": 238,
    "price_unusable": 238,
    "memory_unusable": 543
  },
  "2009": {
    "entities": 456,
    "price_absent": 106,
    "price_unusable": 106,
    "memory_unusable": 427
  },
  "2010": {
    "entities": 569,
    "price_absent": 148,
    "price_unusable": 148,
    "memory_unusable": 477
  },
  "2018": {
    "entities": 466,
    "price_absent": 69,
    "price_unusable": 71,
    "memory_unusable": 244
  },
  "2012": {
    "entities": 719,
    "price_absent": 180,
    "price_unusable": 180,
    "memory_unusable": 432
  },
  "2014": {
    "entities": 973,
    "price_absent": 291,
    "price_unusable": 291,
    "memory_unusable": 197
  },
  "2026": {
    "entities": 569,
    "price_absent": 112,
    "price_unusable": 204,
    "memory_unusable": 458
  },
  "2015": {
    "entities": 819,
    "price_absent": 282,
    "p

## Invalid and unusual values
Invalid rules precede Tukey fences. Positive extremes are retained with flags. Cohorts separate field, unit/currency, and observed OS family. A minimum of 20 distinct resolved entities and nonzero IQR are required. These cross-sectional flags prompt review; they do not prove an error or identify a smartphone. Full flagged examples and fences are in profile.json.

In [4]:
show(p["invalid_examples"])
show(p["robust_fences"])
print("Unusual current cells:",len(p["unusual_examples"]))
for field in ("price","weight","displaysize","batdescription1","internalmemory"):
    rows=[r for r in p["unusual_examples"] if r["field"]==field]
    print(field, "flagged", len(rows))
    show(sorted(rows,key=lambda r:r["number"])[-3:])


[
  {
    "source_id": "2893",
    "name": "Motorola A1210",
    "field": "year",
    "raw": "2009, April. Released 2008, July",
    "flag": "invalid_year"
  },
  {
    "source_id": "10619",
    "name": "Nokia 3 V",
    "field": "year",
    "raw": "2020, August 20. Released 2019, August 23",
    "flag": "invalid_year"
  },
  {
    "source_id": "7098",
    "name": "alcatel Pixi 3 (8) 3G",
    "field": "year",
    "raw": "2015, March. Released 2014, August",
    "flag": "invalid_year"
  },
  {
    "source_id": "4783",
    "name": "Yezz Ritmo 2 YZ420",
    "field": "year",
    "raw": "2012, Q2. Released 2011, July",
    "flag": "invalid_year"
  },
  {
    "source_id": "4784",
    "name": "Yezz Ritmo 3 TV YZ433",
    "field": "year",
    "raw": "2012, Q2. Released 2011, July",
    "flag": "invalid_year"
  }
]
[
  {
    "field": "batdescription1",
    "unit": "mAh",
    "os": "Android",
    "low": -1750.0,
    "high": 9050.0
  },
  {
    "field": "batdescription1",
    "unit": "mAh",
    "o

## Across rows and fields
Inspect every repeated model observation rather than treating identical catalog snapshots as a stuck sensor. The collection contains too little repeated history for supported spike/level-shift detection. Check explicit dates against retrieval and release-year statements against announcement years. Do not invent exact days for month/quarter-only announcements.

In [5]:
show(p["sequence"])
show(p["cross_field"])


{
  "snapshots_per_entity": {
    "1": 14876,
    "5": 1
  },
  "repeated_observations": [
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-09T20:35:09.483447Z",
      "source_field": "batdescription1",
      "value_text": "9000 mAh"
    },
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-09T23:17:50.183664Z",
      "source_field": "batdescription1",
      "value_text": "9000 mAh"
    },
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-10T10:33:08.080093Z",
      "source_field": "batdescription1",
      "value_text": "9000 mAh"
    },
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-10T20:36:11.950248Z",
      "source_field": "batdescription1",
      "value_text": "9000 mAh"
    },
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-10T20:57:41.574622Z",
      "source_field": "batdescription1",
      "value_text": "9000 mAh"
    },
    {
      "phone_key": 2972,
      "retrieved_at": "2026-10-09T20:35:09.483447Z",
      "sou

## Text and identity
Exact IDs are authoritative. NFC, whitespace normalization, explicit Turkish folding, then casefold are used only for comparison. Name collisions and Wikidata multiplicity are review proposals, not merge evidence. Inspect raw text with repr() in distinct-spec-text.csv and phone-text.csv. No fuzzy matches or identity merges are accepted automatically.

In [6]:
show(p["identity"])
print("Full exports: docs/m3/profile.json, distinct-spec-text.csv, phone-text.csv, entity-coverage.csv, entity-field-profile.csv, entity-measurements.csv")


{
  "models_multiple_qids": [
    {
      "source_id": "5418",
      "n": 2
    },
    {
      "source_id": "8584",
      "n": 2
    },
    {
      "source_id": "8062",
      "n": 2
    },
    {
      "source_id": "3390",
      "n": 2
    },
    {
      "source_id": "5137",
      "n": 2
    },
    {
      "source_id": "7673",
      "n": 2
    },
    {
      "source_id": "10316",
      "n": 2
    },
    {
      "source_id": "3115",
      "n": 2
    },
    {
      "source_id": "4238",
      "n": 2
    },
    {
      "source_id": "572",
      "n": 2
    },
    {
      "source_id": "9840",
      "n": 2
    },
    {
      "source_id": "10396",
      "n": 2
    },
    {
      "source_id": "7949",
      "n": 2
    },
    {
      "source_id": "8804",
      "n": 2
    },
    {
      "source_id": "6144",
      "n": 2
    },
    {
      "source_id": "7534",
      "n": 2
    },
    {
      "source_id": "9720",
      "n": 2
    },
    {
      "source_id": "6992",
      "n": 2
    },
    {
      "so